In [1]:
import pandas as pd
import ir_datasets
from src.data import DATA_DIR_INTERIM, DATA_DIR_RAW
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, ROSRBO, binarize_qrels


from src.data import load_qrels_from_path

from topic_gen import logger
logger.setLevel("DEBUG")

In [2]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_INTERIM / "robust-pre-requisites"
predictions, names, metadata = load_qrels_from_path(BASE_DIR)

# binarize qrels
predictions = [binarize_qrels(qrels) for qrels in predictions]

In [3]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 11/2940 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 2/2949 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 5/2946 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 8/2943 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 2/2949 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 4/2947 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 4/2947 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 15/2936 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 6/2945 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Miss

In [4]:
def format_score(row):
    return f"{row['value']:.2f} ± {row['ci']:.2f}"


df = pd.DataFrame(res)
df["score"] = df.apply(format_score, axis=1)
df = df.pivot(index="name", columns="measure", values="score").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")

# Prerequisite
### Can we reproduce the results from Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [ ]:
df[df["prompt"] == "-DNA-zero-shot"][["model", "prompt",
                                      "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qwen3-30B-no-think,-DNA-zero-shot,0.75 ± 0.03,0.11 ± 0.01,0.89 ± 0.01
1,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.02,0.13 ± 0.01,0.85 ± 0.01
2,gpt-4.1,-DNA-zero-shot,0.64 ± 0.02,0.17 ± 0.01,0.81 ± 0.01
9,gpt-oss-120B,-DNA-zero-shot,0.70 ± 0.02,0.14 ± 0.01,0.84 ± 0.01


### Do LLM judges benefit from additional information in the topic?
Binary Setting:
- Für `GPT` scheint mehr Kontext schlechter zu sein. Für `Qwen` scheint mehr Kontext besser zu sein (mit ausnahmen...).
- Genereller ist der Effekt gering

In [ ]:
partial_annotation_prompts = [
    "-DNA-zero-shot", "-DNA-zero-shot-masked-title", "-DNA-zero-shot-masked-description", "-DNA-zero-shot-masked-narrative",
    "-DNA-zero-shot-masked-title-description", "-DNA-zero-shot-masked-title-narrative", "-DNA-zero-shot-masked-description-narrative"]

df["prompt"] = pd.Categorical(df["prompt"], partial_annotation_prompts)
df[df["topics_prompt"] == "human"].sort_values(by=["model", "prompt"])[
    ["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
2,gpt-4.1,-DNA-zero-shot,0.64 ± 0.02,0.17 ± 0.01,0.81 ± 0.01
9,gpt-oss-120B,-DNA-zero-shot,0.70 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
15,gpt-oss-120B,-DNA-zero-shot-masked-title,0.69 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
11,gpt-oss-120B,-DNA-zero-shot-masked-description,0.65 ± 0.02,0.17 ± 0.01,0.82 ± 0.01
12,gpt-oss-120B,-DNA-zero-shot-masked-narrative,0.64 ± 0.02,0.17 ± 0.01,0.82 ± 0.01
13,gpt-oss-120B,-DNA-zero-shot-masked-title-description,0.63 ± 0.02,0.18 ± 0.01,0.81 ± 0.01
14,gpt-oss-120B,-DNA-zero-shot-masked-title-narrative,0.69 ± 0.02,0.15 ± 0.01,0.83 ± 0.01
10,gpt-oss-120B,-DNA-zero-shot-masked-description-narrative,0.70 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
1,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.02,0.13 ± 0.01,0.85 ± 0.01
8,qwen3-14B-no-think,-DNA-zero-shot-masked-title,0.70 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
